<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/ai/02_machine_learning/supervised_learning/classification/breast_cancer_svm_cross_validation/project_breast_cancer_svm_cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CROSS VALIDATION DEMO
# Production-Level ML Workflow
# ============================================================

import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
# ============================================================
# LOAD DATASET
# ============================================================

data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset Shape:", X.shape)

Dataset Shape: (569, 30)


In [3]:
# ============================================================
# SPLIT DATASET
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples :", X_test.shape[0])

Training Samples: 455
Testing Samples : 114


In [4]:
# ============================================================
# PIPELINE
# ============================================================

pipeline = Pipeline([
    ('scaler', StandardScaler()),

    ('svm', SVC())
])

In [5]:
# ============================================================
# STRATIFIED K-FOLD
# ============================================================

kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
# ============================================================
# CROSS VALIDATION
# ============================================================

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=kfold,
    scoring='accuracy'
)

print("\nCross Validation Scores:")
print(cv_scores)

print(f"\nAverage CV Accuracy: {cv_scores.mean():.4f}")

print(f"Standard Deviation: {cv_scores.std():.4f}")


Cross Validation Scores:
[0.94505495 0.97802198 0.96703297 0.96703297 0.98901099]

Average CV Accuracy: 0.9692
Standard Deviation: 0.0146


Hyperparameter Tuning

In [7]:
# ============================================================
# GRID SEARCH CV
# ============================================================

param_grid = {
    'svm__C': [0.1, 1, 10],

    'svm__kernel': ['linear', 'rbf']
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=kfold,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid_search.best_params_)

print(f"\nBest CV Accuracy: {grid_search.best_score_:.4f}")


Best Parameters:
{'svm__C': 0.1, 'svm__kernel': 'linear'}

Best CV Accuracy: 0.9758


In [8]:
# ============================================================
# BEST MODEL
# ============================================================

best_model = grid_search.best_estimator_

best_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svm', SVC(C=0.1, kernel='linear'))])

In [9]:
# ============================================================
# TEST PREDICTIONS
# ============================================================

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 0.9825


In [10]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)


Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [11]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:\n")
print(cm)


Confusion Matrix:

[[41  1]
 [ 1 71]]
